# Sipply — Fine-tune Qwen2.5-7B with LoRA

Fine-tunes Qwen2.5-7B-Instruct to replace GPT-4o-mini for article and cluster summarization.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Upload `finetune_train.jsonl` and `finetune_eval.jsonl` to Colab Files

**Steps:**
1. Install dependencies
2. Load model + apply LoRA
3. Train (~45 min on T4)
4. Evaluate with ROUGE scores
5. Save adapter weights

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────
!pip install -q transformers==4.47.1 peft==0.13.2 trl==0.12.2 \
    bitsandbytes==0.45.0 accelerate==1.2.1 datasets==3.2.0 \
    rouge-score sentencepiece

In [ ]:
# ── Cell 2: Imports & config ───────────────────────────────────────────────
import json
import torch
from pathlib import Path
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer

MODEL_ID   = "Qwen/Qwen2.5-7B-Instruct"
OUTPUT_DIR = "./sipply-qwen-lora"
MAX_SEQ_LEN = 2048

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Cell 3: Load training data ─────────────────────────────────────────────
def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

train_raw = load_jsonl("finetune_train.jsonl")
eval_raw  = load_jsonl("finetune_eval.jsonl")
print(f"Train: {len(train_raw)}  Eval: {len(eval_raw)}")

# Task breakdown
from collections import Counter
print(Counter(r['task'] for r in train_raw))

In [ ]:
# ── Cell 4: Load tokenizer ─────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print("Tokenizer loaded")

In [ ]:
# ── Cell 5: Format to ChatML ───────────────────────────────────────────────
def format_example(row):
    """Convert messages list to a single ChatML string for SFT."""
    return tokenizer.apply_chat_template(
        row["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )

def to_hf_dataset(rows):
    texts = [format_example(r) for r in rows]
    return Dataset.from_dict({"text": texts})

train_ds = to_hf_dataset(train_raw)
eval_ds  = to_hf_dataset(eval_raw)

# Preview one example
print(train_ds[0]["text"][:600])
print("...")

In [ ]:
# ── Cell 6: Load model in 4-bit ────────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False
print("Model loaded (4-bit quantized)")

In [ ]:
# ── Cell 7: Apply LoRA ─────────────────────────────────────────────────────
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,                          # rank — higher = more expressive, more params
    lora_alpha=32,                 # scaling factor
    target_modules=[               # which layers to adapt
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected: ~0.5% of params trainable (~40M out of 7B)

In [ ]:
# ── Cell 8: Training args ──────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,   # effective batch = 8
    per_device_eval_batch_size=2,
    warmup_ratio=0.05,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    bf16=True,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none",
    optim="paged_adamw_8bit",
)

In [ ]:
# ── Cell 9: Train ──────────────────────────────────────────────────────────
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    packing=False,
)

print("Starting training...")
trainer.train()
print("Training complete!")

In [ ]:
# ── Cell 10: Evaluate with ROUGE ───────────────────────────────────────────
from rouge_score import rouge_scorer as rs
import numpy as np

scorer = rs.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

model.eval()
results = []

for row in eval_raw[:50]:   # evaluate on first 50 examples
    messages = row["messages"][:-1]   # system + user, no assistant
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    prediction = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    reference  = row["messages"][-1]["content"]

    # Compare summary field only
    try:
        pred_json = json.loads(prediction)
        ref_json  = json.loads(reference)
        pred_text = pred_json.get("short_summary") or pred_json.get("summary", "")
        ref_text  = ref_json.get("short_summary")  or ref_json.get("summary", "")
    except json.JSONDecodeError:
        pred_text = prediction[:500]
        ref_text  = reference[:500]

    scores = scorer.score(ref_text, pred_text)
    results.append(scores)

print("\n── ROUGE Evaluation (50 eval examples) ──")
for metric in ["rouge1", "rouge2", "rougeL"]:
    f1s = [r[metric].fmeasure for r in results]
    print(f"  {metric:8s}  F1={np.mean(f1s):.4f}  (±{np.std(f1s):.4f})")

In [ ]:
# ── Cell 11: Qualitative check ─────────────────────────────────────────────
test_article = """
Title: OpenAI releases o3-mini with improved reasoning at lower cost

Article text:
OpenAI announced o3-mini today, a new reasoning model that achieves 87.3% on AIME 2024
math benchmarks. The model is available via API at $1.10 per million input tokens,
significantly cheaper than o1. OpenAI says o3-mini is designed for coding, math, and
scientific reasoning tasks where step-by-step thinking is required.
"""

messages = [
    {"role": "system", "content": eval_raw[0]["messages"][0]["content"]},
    {"role": "user",   "content": test_article},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=400, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)
result = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("Model output:")
print(result)

In [ ]:
# ── Cell 12: Save adapter ──────────────────────────────────────────────────
adapter_path = f"{OUTPUT_DIR}/final-adapter"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"\n✓ LoRA adapter saved to {adapter_path}")
print("Download this folder — it's only ~200MB (just the adapter weights, not the full 7B model)")

# List saved files
import os
for f in sorted(os.listdir(adapter_path)):
    size = os.path.getsize(f"{adapter_path}/{f}") / 1e6
    print(f"  {f:40s}  {size:.1f} MB")

## After training

1. **Download** the `sipply-qwen-lora/final-adapter/` folder
2. **Install Ollama** on your Mac: `brew install ollama`
3. **Run inference locally** using the adapter with `transformers` + `peft`:

```python
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-7B-Instruct')
model = PeftModel.from_pretrained(base, './final-adapter')
```

## ROUGE baseline comparison

Run the same eval cells on raw Qwen2.5-7B (without fine-tuning) to get a baseline,
then compare:

| | ROUGE-1 | ROUGE-2 | ROUGE-L |
|---|---|---|---|
| Qwen2.5-7B (base) | TBD | TBD | TBD |
| **Sipply fine-tune** | TBD | TBD | TBD |
| GPT-4o-mini (teacher) | 1.00 | 1.00 | 1.00 |